# Chapter 2 — Agentic Graph Architecture Foundations

Companion notebook to `skills/architecture/`, and the **map for the whole repo**.
This is the *architecture* chapter: no single primitive, but the mental model every
other chapter's skills plug into. Chapter 1 named the crisis (enterprise agents that
stall); this chapter lays the two-graph foundation that fixes it; Chapters 3–8 each
deepen one part of that foundation. Read Section 0 first — it introduces every
component the repo ships and shows how they compose. Then we wire the three
Chapter-2 skills together on the book's running example.

**Scenario (the running DevOps agent).** An SRE is paged at 3:47 a.m.: *high
latency on `checkout-service`*. The tools exist (CloudWatch, X-Ray, logs) but
they are disconnected — "there is no graph, there are only strings." The AWS
account is fictional (`123456789012`); every boto3 call goes through a `moto`
mock, so the notebook runs with zero credentials. This one scenario threads
through **all eight chapter notebooks** — each chapter exercises a different part
of the same latency investigation.

**What this notebook teaches, in order:**
0. *The repo at a glance* — every component (8 chapters, 51 skills, the running
   scenario, the skill anatomy) and the flow of ideas that connects them.
1. *Strings to things* — build a vertical knowledge graph fragment and traverse
   it (Example 2-1 / 2-2).
2. *The dual-graph split* — route an on-call request to the vertical graph, the
   horizontal graph, or **both** (`dual-graph-router`).
3. *The harness* — split the latency investigation into constrained nodes by
   tool surface (`harness-node-splitter`).
4. *The horizontal workflow graph as a DAG* — Example 2-3, where `query_kg`
   feeds two downstream nodes.
5. *The seam* — an execution node touching a `moto`-mocked CloudWatch (Where the
   Two Graphs Meet: the workflow decides which services to check, the knowledge
   graph said which ones they are).
6. *The eight pillars* — map the agent's initial state and read the roadmap
   (`eight-pillar-readiness-map`).

## 0. The repo at a glance — the components this chapter holds together

Everything in this repo is an operational companion to *Agentic GraphRAG*
(O'Reilly). The book carries the theory; the repo is the practice: **51 skills
across 8 chapters, one deep pedagogical notebook per chapter, plus six seam-
validation spike notebooks.** Chapter 2 is where the pieces get their shape,
because the two ideas introduced here — the **dual graph** and the **harness** —
are the substrate every later chapter's skills extend.

### The organizing idea: two graphs + one harness

- **Vertical knowledge graph** — *what is true* about the domain (services,
  dependencies, owners, incidents). "Strings to things." Chapter 3 fills it,
  Chapter 4 makes it durable and temporal.
- **Horizontal workflow graph** — *what the agent does* (a DAG of constrained
  nodes). Chapter 5 plans over it, Chapter 6 executes it as tools.
- **The harness** — the boundary that splits work into nodes by *tool surface*,
  each with a constrained context scope. Chapter 7 lets the harness improve
  itself; Chapter 8 makes it cheap.

Every skill in the repo is a primitive that lives on one of those three
structures. That is the flow of ideas below.

### Flow of ideas — how the eight chapters compose

| # | Chapter | Skills folder | n | What it adds to the foundation |
|---|---------|---------------|---|--------------------------------|
| 1 | Defining Agentic AI | `skills/crisis/` | 5 | Names the failure modes (context, readiness, constraint triangle, workflow-vs-agent, vector-vs-graph) that motivate the whole stack. |
| **2** | **Architecture Foundations** | **`skills/architecture/`** | **3** | **This chapter. The two-graph split (`dual-graph-router`), the harness (`harness-node-splitter`), and the readiness map (`eight-pillar-readiness-map`).** |
| 3 | Knowledge Representation | `skills/knowledge-representation/` | 8 | Fills the *vertical* graph: model selection, three-graph routing, schema patterns, homoiconic meta-schema, capability-authorization, entity resolution, and KG-extraction-approach selection. |
| 4 | Memory | `skills/memory/` | 8 | Makes the vertical graph durable + temporal: bi-temporal edges, incremental update, hierarchical memory, RRF hybrid retrieval, consolidation, and multi-agent memory-consistency selection. |
| 5 | Reasoning & Planning | `skills/reasoning-planning/` | 6 | Plans over the *horizontal* graph: DAG planner, loop-vs-pipeline routing, constraint-guided validation, parallel reconcile-merge, and structured-output contract design (the keystone). |
| 6 | Tool Orchestration | `skills/tool-orchestration/` | 9 | Executes the horizontal graph as tools: RAG-MCP tool selection, MCP gateway, information-flow-control, trust verification, federated governance, CLI-vs-MCP-vs-Skill primitive selection, and the irreversible-action gate. |
| 7 | Self-Evolution & Evaluation | `skills/self-evolution/` | 7 | Closes the loop: execution-graph, four-layer eval cascade, semantic backprop, graduated validation, the self-improving skill object. |
| 8 | Optimization | `skills/optimization/` | 5 | Makes it affordable: model routing, cost-performance scoring, subgraph access control, schema-evolution migration, KV-cache budgeting. |

**Total: 51 skills.** Read a row as "this chapter deepens *that* part of the
Chapter-2 foundation." The three skills demonstrated in Sections 2–7 below are
row 2; every other row is a chapter you can open next.

### How every component is structured

Each `skills/<chapter>/<skill>/` folder is a self-contained, multi-harness primitive:

- **`SKILL.md`** — 7-section anatomy (Overview / When to Use / When NOT /
  Process / Rationalizations / Red Flags / Non-Negotiable Verification) + a
  Security Posture section + a Source Attribution line back to the book chapter.
- **`lib.py`** — pure-Python implementation; production swaps flagged as TODOs at the seam.
- **`cli.py`** — argparse CLI; `--help` prints the SKILL.md description; every
  Process step maps to a subcommand or flag. Runs in Claude Code, Cursor, Gemini
  CLI, Windsurf, OpenCode, or from cron / CI.

And each chapter ships one notebook (`notebooks/chN-*.ipynb`) that exercises its
skills against the *same* `moto`-mocked AWS latency investigation, so the
components are never demonstrated in isolation — they compose on one running
system. The six `notebooks/spike-*.ipynb` files validate the trickiest seams
(bi-temporal edges, incremental Graphiti update, hierarchical memory, Letta
failure modes, the execution graph, RAG-MCP tool selection).

With the map in hand, the rest of this notebook builds row 2 — the foundation
the other seven rows stand on.

## 1. Load the three architecture skills

Each skill ships a `lib.py`. Spike A imported one skill with the `sys.path` +
`import lib` pattern, but all three skills here name their module `lib`, so a
naive import would collide. We load each `lib.py` from its own folder under a
distinct module name via `importlib` — the same "run the skill's library from
its directory" idea, generalized to three skills.

In [ ]:
import sys
import importlib.util
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARCH = PROJECT_ROOT / 'skills' / 'architecture'


def load_skill(slug):
    """Load skills/architecture/<slug>/lib.py under a unique module name."""
    path = ARCH / slug / 'lib.py'
    name = f"{slug.replace('-', '_')}_lib"
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod  # required so dataclasses can resolve annotations
    spec.loader.exec_module(mod)
    return mod


dg = load_skill('dual-graph-router')
splitter = load_skill('harness-node-splitter')
pillars = load_skill('eight-pillar-readiness-map')
print('Loaded architecture skills:')
print(f'  dual-graph-router        from {ARCH / "dual-graph-router"}')
print(f'  harness-node-splitter    from {ARCH / "harness-node-splitter"}')
print(f'  eight-pillar-readiness-map from {ARCH / "eight-pillar-readiness-map"}')

## 2. Strings to things — the vertical knowledge graph

A vector store turns the checkout service, its database, and its config into
floating-point vectors. It can tell you two paragraphs are *similar*; it cannot
tell you `checkout-service` **DEPENDS_ON** `payments-db`. The knowledge graph
makes that relationship an explicit, queryable fact.

Below is the infrastructure fragment from Example 2-2 as a plain-Python graph
(no database needed): typed nodes, typed directed edges, temporal `since`
metadata. Then we answer the question the vector store cannot — *what does
`checkout-service` depend on?* — by **traversing edges**, not ranking text.

In [ ]:
# Example 2-2: a fragment of the infrastructure vertical knowledge graph.
# Nodes carry a type + properties; edges are typed, directed, with temporal metadata.
kg = {
    'nodes': {
        'checkout-service': {'type': 'Service', 'version': '3.2.1', 'status': 'healthy'},
        'user-service':     {'type': 'Service', 'version': '2.1.0', 'status': 'healthy'},
        'payments-db':      {'type': 'Database', 'engine': 'PostgreSQL', 'version': '15.2'},
        'stripe-python':    {'type': 'Library', 'version': '5.4.0'},
    },
    'edges': [
        {'from': 'checkout-service', 'type': 'DEPENDS_ON',  'to': 'payments-db',  'since': '2024-01-15'},
        {'from': 'checkout-service', 'type': 'DEPENDS_ON',  'to': 'user-service', 'since': '2024-03-01'},
        {'from': 'checkout-service', 'type': 'USES_LIBRARY', 'to': 'stripe-python', 'version': '5.4.0'},
    ],
}


def traverse(graph, start, edge_type=None):
    """One-hop traversal: follow (optionally typed) edges out of `start`.
    This is the graph query from Example 2-1 as a stdlib function — an explicit
    relationship traversal, not a similarity search."""
    out = []
    for e in graph['edges']:
        if e['from'] == start and (edge_type is None or e['type'] == edge_type):
            out.append((e['type'], e['to'], {k: v for k, v in e.items()
                                             if k not in ('from', 'to', 'type')}))
    return out


deps = traverse(kg, 'checkout-service')
print('checkout-service is connected to:')
for etype, target, props in deps:
    print(f'  -[{etype}]-> {target}   {props}')

# The relationships are explicit facts, not statistical inferences.
assert ('DEPENDS_ON', 'payments-db', {'since': '2024-01-15'}) in deps
assert any(t == 'stripe-python' for _, t, _ in deps)
print('\nPASS: dependency chain is a traversal, not a text-similarity guess.')

## 3. Route the on-call request — `dual-graph-router`

Before doing any work, the harness must decide **what kind** of request this is.
"The vertical graph is the map. The horizontal graph is the route."

- A pure lookup ("which services depend on payments-db?") is one traversal →
  **vertical**.
- A pure process ("classify, plan, decide whether to roll back") → **horizontal**.
- A diagnosis that needs domain facts ("why is checkout slow — what does it
  depend on and what changed?") is a workflow whose retrieval node traverses the
  vertical graph and writes the result back → **both**. This is *Where the Two
  Graphs Meet*, and it is where the architecture earns its keep.

In [ ]:
requests = [
    'Which services depend on the payments-db and were deployed in the last 24 hours?',
    'Classify the alert severity, plan the remediation, and decide whether to roll back.',
    'Diagnose why the checkout API is slow: which services does it depend on and what changed recently?',
]

decisions = dg.route_batch(requests)
for d in decisions:
    print(f'[{d.target.upper():<10}] {d.request}')
    print(f'             vertical={d.vertical_score} horizontal={d.horizontal_score} '
          f'(V:{d.matched_vertical}  H:{d.matched_horizontal})')

lookup, process, diagnosis = decisions
assert lookup.target == 'vertical'
assert process.target == 'horizontal'
assert diagnosis.target == 'both'

# The 'both' route must explain the bidirectional interaction.
meet = dg.explain_meeting_point(diagnosis)
print('\nWhere the two graphs meet (for the diagnosis request):')
for role, text in meet.items():
    print(f'  {role}: {text}')
assert meet['forward_flow'] and meet['backward_flow']
print('\nPASS: lookup->vertical, process->horizontal, diagnosis->both with a meeting point.')

## 4. Split the latency investigation into harness nodes — `harness-node-splitter`

The diagnosis routed to a horizontal workflow. Now we split that workflow into
nodes. The rule from the chapter: **nodes differ by tool surface, not by prompt.**
Two operations that call the same tools are one node with prompt variation; two
that call different tools are different roles (the RedAI scanner-vs-validator
lesson).

The bundled sample workflow contains the Example 2-3 nodes plus a deliberate
merge pair (`get_metrics` + `get_cpu`, identical tool surface) and the RedAI
split pair (`scan_code` filesystem vs `validate_finding` browser/network).

In [ ]:
import json

sample = ARCH / 'harness-node-splitter' / 'sample-workflow.json'
rows = json.loads(sample.read_text(encoding='utf-8'))['operations']
ops = splitter.operations_from_dicts(rows)
result = splitter.split_nodes(ops)

print(f'{len(ops)} candidate operations -> {len(result.nodes)} nodes '
      f'(tool-overlap threshold {result.threshold})\n')
for n in result.nodes:
    tag = '  [MERGED]' if n.is_merged else ''
    print(f'  {n.id:<18}{n.node_type:<11}tools={n.tools}{tag}')
    if n.is_merged:
        print(f'      prompt variants: {n.tasks}')

# The two metric-query ops share a tool surface -> one node with two prompts.
merged = [n for n in result.nodes if n.is_merged]
assert len(merged) == 1 and set(merged[0].merged_from) == {'get_metrics', 'get_cpu'}

# RedAI: disjoint tool surfaces -> stay split.
assert splitter.tool_overlap({'filesystem'},
                             {'browser_driver', 'ios_simulator', 'network_stack', 'scripting_runtime'}) == 0.0
ids = {n.id for n in result.nodes}
assert {'scan_code', 'validate_finding'} <= ids
print('\nPASS: same tool surface merged; distinct tool surfaces stayed split.')

### The per-node constrained context scope

Splitting is not just boundary-drawing — each node gets a **constrained context
scope**: the tool surface it may invoke, the memory slices it reads and writes,
and the input/output contract the schema validator holds it to (three of the
harness's six surfaces). A node sees only its own tools and its own memory slice,
not the whole registry or the whole graph. That is least privilege by
construction.

In [ ]:
retrieval_node = next(n for n in result.nodes if n.id == 'query_kg')
scope = splitter.node_scope(retrieval_node)
print('Scope of the retrieval node that traverses the vertical graph:')
for k, v in scope.items():
    print(f'  {k}: {v}')

assert scope['tool_surface'] == ['graph_read']          # only the graph, nothing else
assert 'vertical_kg' in scope['memory_reads']
print('\nPASS: the retrieval node is scoped to graph_read + the vertical_kg memory slice.')

## 5. The horizontal workflow graph as a DAG (Example 2-3)

The nodes form a DAG: edges are dependencies, not a linear sequence. `query_kg`
feeds **both** `get_metrics` and `analyze`; `analyze` waits for the graph context
*and* the live metrics before it runs. Independent branches can run in parallel;
dependent ones wait. We verify the structure is an actual DAG (no cycle) and that
`analyze` has two inputs.

In [ ]:
# Example 2-3: the latency-investigation workflow graph.
investigation = {
    'nodes': ['classify', 'query_kg', 'get_metrics', 'analyze', 'report'],
    'edges': [
        ('classify', 'query_kg'),      # severity determines scope of traversal
        ('query_kg', 'get_metrics'),    # dependencies determine which metrics to check
        ('query_kg', 'analyze'),        # graph context feeds root-cause analysis
        ('get_metrics', 'analyze'),     # live metrics feed root-cause analysis
        ('analyze', 'report'),          # analysis becomes the incident report
    ],
}


def is_dag_and_indegrees(g):
    """Kahn's algorithm: returns (is_dag, indegree_map). If we can remove every
    node in dependency order, there is no cycle."""
    indeg = {n: 0 for n in g['nodes']}
    for _, dst in g['edges']:
        indeg[dst] += 1
    original = dict(indeg)
    ready = [n for n, d in indeg.items() if d == 0]
    removed = 0
    work = dict(indeg)
    while ready:
        n = ready.pop()
        removed += 1
        for src, dst in g['edges']:
            if src == n:
                work[dst] -= 1
                if work[dst] == 0:
                    ready.append(dst)
    return removed == len(g['nodes']), original


is_dag, indeg = is_dag_and_indegrees(investigation)
print('in-degree per node (how many inputs each waits on):')
for n in investigation['nodes']:
    print(f'  {n:<12} waits on {indeg[n]} input(s)')

assert is_dag, 'the workflow graph must be acyclic'
assert indeg['analyze'] == 2, 'analyze waits on both query_kg and get_metrics'
print('\nPASS: valid DAG; analyze waits on two inputs (graph context + live metrics).')

## 6. The seam — an execution node touching moto-mocked CloudWatch

*Where the Two Graphs Meet* in action: the workflow graph decided we need live
metrics for the affected services; the **vertical graph already told us which
services those are** (`checkout-service`, and its dependency `payments-db`). The
`get_metrics` execution node now calls the real boto3 CloudWatch surface —
mocked by `moto` against fictional account `123456789012`, so no credentials and
no charges. Remove `@mock_aws` and the `AWS_*` env vars to target a real account.

In [ ]:
import os
import datetime as dt

os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'testing')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'testing')

import boto3
from moto import mock_aws

FICTIONAL_ACCOUNT_ID = '123456789012'

# Which services to query is NOT guessed — it comes from the vertical-graph traversal.
affected = ['checkout-service'] + [t for _, t, _ in traverse(kg, 'checkout-service', 'DEPENDS_ON')]
print(f'Vertical graph says the affected surface is: {affected}')


@mock_aws
def get_metrics_for(services):
    cw = boto3.client('cloudwatch')
    now = dt.datetime.now(dt.timezone.utc)
    # Seed a latency metric that mirrors the 3:47 a.m. spike for checkout-service.
    for i, minutes in enumerate(range(10, 0, -2)):
        cw.put_metric_data(
            Namespace='DevOps/Checkout',
            MetricData=[{
                'MetricName': 'p99_latency_ms',
                'Dimensions': [{'Name': 'Service', 'Value': 'checkout-service'}],
                'Timestamp': now - dt.timedelta(minutes=minutes),
                'Value': 42.0 if i < 2 else 3100.0,  # spike after the deploy
                'Unit': 'Milliseconds',
            }],
        )
    resp = cw.get_metric_data(
        MetricDataQueries=[{
            'Id': 'p99',
            'MetricStat': {
                'Metric': {
                    'Namespace': 'DevOps/Checkout',
                    'MetricName': 'p99_latency_ms',
                    'Dimensions': [{'Name': 'Service', 'Value': 'checkout-service'}],
                },
                'Period': 60,
                'Stat': 'Maximum',
            },
            'ReturnData': True,
        }],
        StartTime=now - dt.timedelta(minutes=15),
        EndTime=now,
    )
    series = resp['MetricDataResults'][0]
    return {
        'service_queried': 'checkout-service',
        'label': series['Label'],
        'datapoints': len(series['Values']),
        'peak_latency_ms': max(series['Values']) if series['Values'] else None,
        'fictional_account': FICTIONAL_ACCOUNT_ID,
    }


metrics = get_metrics_for(affected)
print('\nboto3 + moto CloudWatch result (the get_metrics execution node):')
for k, v in metrics.items():
    print(f'  {k}: {v}')

# The mocked call must return shaped data the analyze node can consume.
assert metrics['datapoints'] > 0, 'CloudWatch get_metric_data returned no datapoints'
assert metrics['peak_latency_ms'] and metrics['peak_latency_ms'] > 1000, 'the seeded spike should be visible'
print('\nPASS: the execution node reached a moto-mocked AWS seam and got shaped metric data.')

## 7. Map the agent across the eight pillars — `eight-pillar-readiness-map`

The dual-graph architecture is the framework; the eight pillars are the build
roadmap, and they are **layered** — knowledge representation must come first
because everything depends on the graph. The DevOps agent's Chapter-2 starting
point is a *representation problem, not a technology problem*: the data exists in
logs and configs (knowledge representation only **partial**), and every other
pillar is missing. The map reads back which of the five Chapter-1 flaws are still
open and what to build next.

In [ ]:
report = pillars.assess(pillars.initial_state())
print(f'Readiness: {report.readiness_pct}%   next pillar to build: {report.next_pillar}\n')
for p in report.per_pillar:
    mark = {'present': '[x]', 'partial': '[~]', 'missing': '[ ]'}[p['status']]
    print(f"  {mark} {p['order']}. {p['name']:<30}{p['chapter']:<8}solves: {', '.join(p['solves_flaws'])}")

print('\nUnresolved Chapter-1 flaws:')
for f in report.unresolved_flaws:
    print(f"  - {f['flaw']} ({f['state']}) — needs {', '.join(f['solving_pillars'])}")

print(f"\nRoadmap: {' -> '.join(report.roadmap)}")

# The initial state is the representation problem: KR partial, next = KR; all five flaws open.
assert report.next_pillar == 'knowledge_representation'
assert {f['flaw'] for f in report.unresolved_flaws} == set(pillars.FLAW_TO_PILLARS)
assert not report.dependency_violations
print('\nPASS: initial state maps to a representation problem; roadmap starts at Chapter 3.')

## 8. Verdict

Every Chapter-2 skill round-tripped on the running DevOps example, and every
verification gate passed. The final cell asserts the whole chain once more.

| Gate | What it proves |
|------|----------------|
| Vertical-graph traversal returns the explicit dependency chain | strings → things |
| `dual-graph-router` splits lookup / process / diagnosis correctly | the dual-graph distinction + Where the Two Graphs Meet |
| `harness-node-splitter` merges same-surface, splits distinct-surface | nodes differ by tool surface, not prompt |
| Example 2-3 is a valid DAG; `analyze` waits on two inputs | the horizontal workflow graph |
| `get_metrics` reaches moto-mocked CloudWatch with shaped data | the execution seam |
| `eight-pillar-readiness-map` maps the initial state + roadmap | the eight pillars, layered |

In [ ]:
checks = {
    'vertical traversal': any(t == 'payments-db' for _, t, _ in traverse(kg, 'checkout-service')),
    'router: diagnosis -> both': dg.route(requests[2]).target == 'both',
    'splitter: 9 ops -> 8 nodes': len(splitter.split_nodes(ops).nodes) == 8,
    'workflow is a DAG': is_dag_and_indegrees(investigation)[0],
    'moto seam shaped data': metrics['datapoints'] > 0,
    'pillars: next = knowledge_representation': pillars.assess(pillars.initial_state()).next_pillar == 'knowledge_representation',
}
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(checks.values()), 'one or more Chapter-2 gates failed'
print('\nCh2 architecture foundations: all gates passed. On to Chapter 3 — building the knowledge graph.')